# 02 — Feature Engineering

This notebook performs:
1. **Feature-to-feature Pearson correlation** to remove redundant features (|r| >= 0.95)
2. **Feature-to-target Pearson correlation** to identify and drop features with no linear relationship to any malware category
3. **PCA (Principal Component Analysis)** for dimensionality reduction and feature extraction

Outputs are saved to `../data/processed/` for use in notebook 03.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", 60)
sns.set_style("whitegrid")

## 1. Load and Prepare Data

Replicate the cleaning and split from notebook 01 so this notebook is self-contained.

In [ ]:
# Load both CSVs and combine
df_benign = pd.read_csv("../data/raw/All Benign Samples-combined.csv")
df_malware = pd.read_csv("../data/raw/All Malware Samples-Combined2006-2021.csv")
df = pd.concat([df_benign, df_malware], ignore_index=True)
df.columns = df.columns.str.strip()

# Extract category and save Year before dropping metadata
df["category"] = df["Category"].apply(lambda x: x.split("-")[0])
sample_year = df["Year"].copy()
df.drop(columns=["Category", "Year", "Month"], inplace=True)

# Separate features and target
feature_cols = [c for c in df.columns if c not in ["category", "Class"]]
X = df[feature_cols]
y = df["category"]

# === Split 1: Random stratified (same as notebook 01) ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# === Split 2: Temporal (train pre-2020, test 2020-2021) ===
is_benign = sample_year.isna()
is_train_malware = sample_year < 2020
is_test_malware = sample_year >= 2020

benign_idx = df.index[is_benign]
benign_train_idx, benign_test_idx = train_test_split(
    benign_idx, test_size=0.2, random_state=42
)

temporal_train_idx = df.index[is_train_malware].append(benign_train_idx)
temporal_test_idx = df.index[is_test_malware].append(benign_test_idx)

X_train_temp = X.loc[temporal_train_idx]
X_test_temp = X.loc[temporal_test_idx]
y_train_temp = y.loc[temporal_train_idx]
y_test_temp = y.loc[temporal_test_idx]

print(f"Features: {X.shape[1]}")
print(f"\nRandom split  — Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Temporal split — Train: {len(X_train_temp)} | Test: {len(X_test_temp)}")

## 2. Pearson Correlation — Feature-to-Feature Redundancy

Identify highly correlated feature pairs (|r| >= 0.95). When two features carry nearly identical information, keeping both adds noise without predictive value.

In [ ]:
corr = X_train.corr(method="pearson")

# Full correlation heatmap
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1,
            xticklabels=True, yticklabels=True, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Pearson Correlation Matrix — All Features", fontsize=14)
ax.tick_params(axis="both", labelsize=6)
plt.tight_layout()
plt.show()

In [ ]:
# Find highly correlated pairs (|r| >= 0.95)
threshold = 0.95
high_corr_pairs = []

for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        if abs(corr.iloc[i, j]) >= threshold:
            high_corr_pairs.append((
                corr.columns[i], corr.columns[j], round(corr.iloc[i, j], 4)
            ))

high_corr_df = pd.DataFrame(high_corr_pairs, columns=["Feature_1", "Feature_2", "Correlation"])
high_corr_df = high_corr_df.sort_values("Correlation", key=abs, ascending=False).reset_index(drop=True)

print(f"Pairs with |r| >= {threshold}: {len(high_corr_df)}")
high_corr_df

In [ ]:
# Drop one feature from each highly correlated pair
# Strategy: drop the feature with higher mean abs correlation to all others (more redundant)

to_drop = set()
for _, row in high_corr_df.iterrows():
    f1, f2 = row["Feature_1"], row["Feature_2"]
    if f1 in to_drop or f2 in to_drop:
        continue
    mean_corr_f1 = corr[f1].drop(f1).abs().mean()
    mean_corr_f2 = corr[f2].drop(f2).abs().mean()
    if mean_corr_f1 >= mean_corr_f2:
        to_drop.add(f1)
    else:
        to_drop.add(f2)

print(f"Features to drop ({len(to_drop)}): {sorted(to_drop)}")
print(f"Features remaining: {X_train.shape[1] - len(to_drop)}")

In [ ]:
# Apply the filter to both splits
X_train_filtered = X_train.drop(columns=to_drop)
X_test_filtered = X_test.drop(columns=to_drop)
X_train_temp_filtered = X_train_temp.drop(columns=to_drop)
X_test_temp_filtered = X_test_temp.drop(columns=to_drop)

print(f"Shape after correlation filter: {X_train_filtered.shape}")

# Show the cleaned correlation matrix
corr_filtered = X_train_filtered.corr(method="pearson")
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_filtered, cmap="coolwarm", center=0, vmin=-1, vmax=1,
            xticklabels=True, yticklabels=True, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Pearson Correlation Matrix — After Filtering (|r| < 0.95)", fontsize=14)
ax.tick_params(axis="both", labelsize=7)
plt.tight_layout()
plt.show()

## 3. Pearson Correlation — Feature-to-Target Relevance

Measure how strongly each feature correlates with the target classes. One-hot encode `category` (Benign, Ransomware, Spyware, Trojan), compute Pearson |r| between each feature and each class column, then take the **max** — this captures features that distinguish *any* class, not just one.

Features with near-zero max |r| have no linear relationship with any category and are candidates for removal.

In [ ]:
# One-hot encode multi-class target
dummies = pd.get_dummies(y_train).reindex(X_train_filtered.index)

# Max |r| across all classes for each feature
feat_target_corr = X_train_filtered.apply(
    lambda col: dummies.corrwith(col).abs().max()
).sort_values(ascending=False)

# Also store per-class correlations for the heatmap
per_class_corr = pd.DataFrame({
    cls: X_train_filtered.corrwith(dummies[cls])
    for cls in dummies.columns
})

# Display ranked results
print("Feature relevance (max |r| across all classes):\n")
print(feat_target_corr.to_string())
print(f"\nMost relevant:  {feat_target_corr.idxmax()} (|r|={feat_target_corr.max():.4f})")
print(f"Least relevant: {feat_target_corr.idxmin()} (|r|={feat_target_corr.min():.4f})")

In [ ]:
# Heatmap: per-class Pearson correlation for all features
fig, ax = plt.subplots(figsize=(6, 14))
sns.heatmap(per_class_corr.loc[feat_target_corr.index], cmap="coolwarm", center=0,
            vmin=-0.5, vmax=0.5, annot=True, fmt=".2f", ax=ax,
            cbar_kws={"shrink": 0.6})
ax.set_title("Feature–Target Pearson Correlation (per class)", fontsize=13)
ax.set_xlabel("Category")
plt.tight_layout()
plt.show()

In [ ]:
# Horizontal bar chart — ranked by max |r|
fig, ax = plt.subplots(figsize=(10, 12))
feat_target_corr.sort_values().plot.barh(ax=ax, color="steelblue")
ax.axvline(x=0.05, color="red", linestyle="--", label="Threshold (|r| = 0.05)")
ax.set_xlabel("Max |Pearson r| across all classes")
ax.set_title("Feature Relevance to Target — Ranked by Correlation")
ax.legend()
plt.tight_layout()
plt.show()

# List features below threshold
low_relevance = feat_target_corr[feat_target_corr < 0.05]
print(f"\nFeatures with max |r| < 0.05 ({len(low_relevance)}):")
for feat, r in low_relevance.items():
    print(f"  {feat}: {r:.4f}")

In [ ]:
# Drop features with near-zero target correlation (max |r| < 0.05)
low_relevance_cols = feat_target_corr[feat_target_corr < 0.05].index.tolist()

if low_relevance_cols:
    X_train_filtered = X_train_filtered.drop(columns=low_relevance_cols)
    X_test_filtered = X_test_filtered.drop(columns=low_relevance_cols)
    X_train_temp_filtered = X_train_temp_filtered.drop(columns=low_relevance_cols)
    X_test_temp_filtered = X_test_temp_filtered.drop(columns=low_relevance_cols)
    print(f"Dropped {len(low_relevance_cols)} low-relevance features: {low_relevance_cols}")
else:
    print("No features below threshold — all retained.")

print(f"Features remaining: {X_train_filtered.shape[1]}")

## 4. Feature Scaling

Scale the filtered features using `RobustScaler` before PCA (PCA is sensitive to feature magnitudes).

In [ ]:
# Scale random split
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_filtered),
    columns=X_train_filtered.columns, index=X_train_filtered.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_filtered),
    columns=X_test_filtered.columns, index=X_test_filtered.index
)

# Scale temporal split (fit on temporal train only)
scaler_temp = RobustScaler()
X_train_temp_scaled = pd.DataFrame(
    scaler_temp.fit_transform(X_train_temp_filtered),
    columns=X_train_temp_filtered.columns, index=X_train_temp_filtered.index
)
X_test_temp_scaled = pd.DataFrame(
    scaler_temp.transform(X_test_temp_filtered),
    columns=X_test_temp_filtered.columns, index=X_test_temp_filtered.index
)

print(f"Scaled shape (random):   {X_train_scaled.shape}")
print(f"Scaled shape (temporal): {X_train_temp_scaled.shape}")

## 5. PCA — Feature Extraction

Apply PCA to reduce dimensionality. First, fit on all components to find the optimal number via explained variance, then extract the final reduced set.

In [ ]:
# Fit PCA on all components to analyze explained variance
pca_full = PCA(random_state=42)
pca_full.fit(X_train_scaled)

cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_, color="steelblue", alpha=0.8)
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Explained Variance Ratio")
axes[0].set_title("Scree Plot")

# Cumulative variance
axes[1].plot(range(1, len(cumulative_var) + 1), cumulative_var, "o-", color="steelblue")
axes[1].axhline(y=0.95, color="red", linestyle="--", label="95% threshold")
axes[1].axhline(y=0.99, color="orange", linestyle="--", label="99% threshold")
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Explained Variance")
axes[1].set_title("Cumulative Explained Variance")
axes[1].legend()

plt.tight_layout()
plt.show()

# Report thresholds
n_95 = np.argmax(cumulative_var >= 0.95) + 1
n_99 = np.argmax(cumulative_var >= 0.99) + 1
print(f"Components for 95% variance: {n_95}")
print(f"Components for 99% variance: {n_99}")

In [ ]:
# Apply PCA with 95% explained variance — random split
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

pca_cols = [f"PC{i+1}" for i in range(X_train_pca.shape[1])]
X_train_pca_df = pd.DataFrame(X_train_pca, columns=pca_cols, index=X_train_scaled.index)
X_test_pca_df = pd.DataFrame(X_test_pca, columns=pca_cols, index=X_test_scaled.index)

# Apply PCA — temporal split (fit on temporal train)
pca_temp = PCA(n_components=0.95, random_state=42)
X_train_temp_pca = pca_temp.fit_transform(X_train_temp_scaled)
X_test_temp_pca = pca_temp.transform(X_test_temp_scaled)

pca_temp_cols = [f"PC{i+1}" for i in range(X_train_temp_pca.shape[1])]
X_train_temp_pca_df = pd.DataFrame(X_train_temp_pca, columns=pca_temp_cols, index=X_train_temp_scaled.index)
X_test_temp_pca_df = pd.DataFrame(X_test_temp_pca, columns=pca_temp_cols, index=X_test_temp_scaled.index)

print(f"Random split:   {X_train_scaled.shape[1]} -> {X_train_pca_df.shape[1]} components ({pca.explained_variance_ratio_.sum():.4f} variance)")
print(f"Temporal split:  {X_train_temp_scaled.shape[1]} -> {X_train_temp_pca_df.shape[1]} components ({pca_temp.explained_variance_ratio_.sum():.4f} variance)")

In [ ]:
# Visualize first 2 principal components colored by class
fig, ax = plt.subplots(figsize=(10, 7))
categories = sorted(y_train.unique())
colors = plt.cm.Set2(np.linspace(0, 1, len(categories)))

for cat, color in zip(categories, colors):
    mask = y_train == cat
    ax.scatter(X_train_pca_df.loc[mask, "PC1"],
               X_train_pca_df.loc[mask, "PC2"],
               label=cat, alpha=0.4, s=10, color=color)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title("PCA — First Two Principal Components")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Top feature loadings for first 3 PCs
loadings = pd.DataFrame(
    pca.components_.T,
    columns=pca_cols,
    index=X_train_scaled.columns
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, ax in enumerate(axes):
    pc = f"PC{i+1}"
    top = loadings[pc].abs().nlargest(10)
    loadings[pc].loc[top.index].sort_values().plot.barh(ax=ax, color="steelblue")
    ax.set_title(f"{pc} — Top 10 Loadings")
    ax.set_xlabel("Loading")

plt.tight_layout()
plt.show()

## 6. Encode Labels and Save Outputs

Save three feature sets for model comparison in notebook 03:
1. **Correlation-filtered** (scaled, redundant + low-relevance features removed)
2. **PCA-transformed** (dimensionality-reduced)
3. **Labels** (encoded)

In [ ]:
import joblib
import os

# Encode labels for both splits
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

le_temp = LabelEncoder()
y_train_temp_encoded = le_temp.fit_transform(y_train_temp)
y_test_temp_encoded = le_temp.transform(y_test_temp)

print("Label mapping:")
for label, idx in zip(le.classes_, range(len(le.classes_))):
    print(f"  {label} -> {idx}")

# Save both splits to separate subdirectories
for split_name, data in [
    ("random", {
        "X_train_filt": X_train_scaled, "X_test_filt": X_test_scaled,
        "X_train_pca": X_train_pca_df, "X_test_pca": X_test_pca_df,
        "y_train_enc": y_train_encoded, "y_test_enc": y_test_encoded,
        "y_train_idx": y_train.index, "y_test_idx": y_test.index,
        "scaler": scaler, "pca": pca, "le": le,
    }),
    ("temporal", {
        "X_train_filt": X_train_temp_scaled, "X_test_filt": X_test_temp_scaled,
        "X_train_pca": X_train_temp_pca_df, "X_test_pca": X_test_temp_pca_df,
        "y_train_enc": y_train_temp_encoded, "y_test_enc": y_test_temp_encoded,
        "y_train_idx": y_train_temp.index, "y_test_idx": y_test_temp.index,
        "scaler": scaler_temp, "pca": pca_temp, "le": le_temp,
    }),
]:
    out = f"../data/processed/{split_name}"
    os.makedirs(out, exist_ok=True)

    data["X_train_filt"].to_csv(f"{out}/X_train_filtered.csv", index=True)
    data["X_test_filt"].to_csv(f"{out}/X_test_filtered.csv", index=True)
    data["X_train_pca"].to_csv(f"{out}/X_train_pca.csv", index=True)
    data["X_test_pca"].to_csv(f"{out}/X_test_pca.csv", index=True)

    pd.Series(data["y_train_enc"], index=data["y_train_idx"], name="label").to_csv(f"{out}/y_train.csv", index=True)
    pd.Series(data["y_test_enc"], index=data["y_test_idx"], name="label").to_csv(f"{out}/y_test.csv", index=True)

    joblib.dump(data["scaler"], f"{out}/scaler.pkl")
    joblib.dump(data["pca"], f"{out}/pca.pkl")
    joblib.dump(data["le"], f"{out}/label_encoder.pkl")

    print(f"\n{split_name}/ — Filtered: {data['X_train_filt'].shape[1]} cols, PCA: {data['X_train_pca'].shape[1]} cols")

# Save shared artifacts
joblib.dump(sorted(to_drop), "../data/processed/dropped_corr_features.pkl")
print("\nDone.")